In [1]:
# to allow external code modification reload
%load_ext autoreload
%autoreload 2

# Setup

In [2]:
!uv add gitsource


Resolved 129 packages in 5ms
Audited 125 packages in 8ms


# Preparation

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Q1. How many lesson pages


In [5]:
print(len(documents))

72


answer: 72

# Q2. Indexing and searching

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"


search_results = index.search(
    question,
    # boost_dict={"question": 2.0, "section": 0.5},
    # filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [9]:
search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

answer: 01-agentic-rag/lessons/14-agentic-loop.md

# Q3. RAG

In [10]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [11]:
from rag_helper_hw import RAGBase
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer: roughly 7 000 

In [12]:
# added a print in the RAG helper to get the usage.input_tokens 
# (only text answer is given by the rag method)
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

response.usage.input_tokens
6938


# Q4. Chunking

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

answer: 295 chunks

In [15]:
# print(chunks[0])

# Q5. RAG with chunking

In [16]:
from minsearch import Index

index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

In [17]:

assistant_chunks = RAGBase(
    index=index_chunks,
    llm_client=openai_client,
)

In [18]:
# added a print in the RAG helper to get the usage.input_tokens 
# (only text answer is given by the rag method)
answer = assistant_chunks.rag("How does the agentic loop keep calling the model until it stops?")

response.usage.input_tokens
2177


answer: 3 times fewer (7 k = 3x 2k)

# Q6. Turning it into an agent

In [19]:
!uv add toyaikit

Resolved 129 packages in 3ms
Audited 125 packages in 5ms


In [20]:
def search(query: str) -> dict[str, str]:
    """
    Search the LLM Zoomcamp lessons pages for entries matching the given query.
    """
    return index_chunks.search(
        query,
        num_results=5
        # boost_dict={"question": 3.0, "section": 0.5},
        # filter_dict={"course": "llm-zoomcamp"}
    )

In [31]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search)

In [32]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the LLM Zoomcamp lessons pages for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [59]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

""".strip()

prompt = """How does the agentic loop work, and how is it different from plain RAG?
"""

In [60]:
# openai_client = OpenAI(
#     api_key=os.getenv("GITHUB_PAT_TOKEN"),
#     base_url="https://models.inference.ai.azure.com",
# )


In [61]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [62]:
# from dotenv import load_dotenv

# load_dotenv()

# openai_client = OpenAI(
#     api_key=os.getenv("GEMINI_API_KEY"),
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )


In [63]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(client=openai_client, 
                            # model="gpt-4o-mini"     # github
                            # model="groq/compound" # no function calling     
                            # model="gemini-2.5-flash"          # no responses interface
                            # model="llama-3.3-70b-versatile"   # Groq
                            model="openai/gpt-oss-120b"       # Groq
                            )
)

In [64]:
# chat_interface = IPythonChatInterface()
# callback = DisplayingRunnerCallback(chat_interface)

# runner = OpenAIResponsesRunner(
#     tools=agent_tools,
#     developer_prompt=instructions,
#     chat_interface=chat_interface,
#     llm_client=OpenAIClient(client=openai_client, 
#                             model="gpt-4o-mini"     # github
#                             # model="groq/compound" # no function calling     
#                             # model="gemini-2.5-flash"          # no responses interface
#                             # model="llama-3.3-70b-versatile"   # Groq
#                             # model="openai/gpt-oss-120b"       # Groq
#                             )
# )

In [65]:
import warnings
warnings.filterwarnings("ignore", message=".*No pricing data.*")

In [66]:
result = runner.loop(
    prompt=prompt,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01khtmap9rfweth7depvzwcm3p` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8021, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}